# 01 · Case data: download, parse, audit, clean
**Book chapters 4–5.** Output: `data/processed/cases_weekly.csv` (one row per RDHS area per week).

In [ ]:
# ▶ Run this cell first in every notebook (about 1 minute)
from google.colab import drive
drive.mount('/content/drive')
import os, sys
ROOT = os.environ.get('LW_ROOT', '.')
CODE = f'{ROOT}/code/leptowatch'
os.environ['LW_DATA_ROOT'] = f'{ROOT}/data'
os.environ['PYTHONPATH'] = f'{CODE}/src'
sys.path.insert(0, f'{CODE}/src')
os.chdir(CODE)
!apt-get -qq install -y poppler-utils > /dev/null
%pip -q install -r requirements.txt
print('code:', CODE, '| data:', os.environ['LW_DATA_ROOT'])

## Step 1 — the archive copy (1,014 PDFs, about 5 minutes)

In [ ]:
!python -m leptowatch.data.wer_download archive

## Step 2 — newer issues from the official site (dry run first, then real)

In [ ]:
!python -m leptowatch.data.wer_download update --dry-run

In [ ]:
!python -m leptowatch.data.wer_download update

## Step 3 — parse every PDF (about 3–6 minutes)

In [ ]:
from leptowatch.config import path
PDF = str(path('raw','wer_pdfs','x').parent); INTERIM = str(path('interim','x').parent)
!python -m leptowatch.data.wer_parse "{PDF}" "{INTERIM}" --workers 2

## Step 4 — audit
Expect about 908 accepted reports (749 exact + 159 national-cell cut off) of 1,014, before the new issues.

In [ ]:
!python -m leptowatch.data.wer_audit "{INTERIM}"

In [ ]:
import pandas as pd
q = pd.read_csv(f'{INTERIM}/review_queue.csv'); print(len(q), 'reports to review'); q.head(20)

## Step 5 — build the clean case panel
Week fixes go in `config/week_fixes.csv` (pdf, year, epi_week) after you check a PDF by eye; hand-check
corrections go in `config/manual_overrides.csv`.

In [ ]:
!python -m leptowatch.data.cases "{INTERIM}" "{path('processed','cases_weekly.csv')}"

In [ ]:
c = pd.read_csv(path('processed','cases_weekly.csv'), parse_dates=['week_start'])
nat = c.groupby('week_start').cases.sum(min_count=20)
ax = nat.plot(figsize=(12,3), title='Reported leptospirosis cases per week, all RDHS areas'); ax.set_ylabel('cases')

## Step 6 — the hand check (E5)
Make a random sample of 60 rows, open each PDF and compare. Record differences in `config/manual_overrides.csv`.

In [ ]:
r = pd.read_csv(f'{INTERIM}/wer_rows.csv')
s = r[~r.rdhs.str.startswith('SRI')].sample(60, random_state=7)[['pdf','page','rdhs','cases_week','cases_ytd','raw_line']]
s['pdf_A_week'] = ''; s['pdf_B_ytd'] = ''; s['match_Y_N'] = ''; s['checked_by'] = ''
s.to_csv(path('reports','hand_check_sheet.csv'), index=False); s.head()